# Fase 1: modelo predictivo de la calificación de películas

Proyecto integrador de Modelos y Simulación de Sistemas I (Universidad de Antioquia).
Equipo: Alejandro Múnera Ramírez, Miguel Ángel Altamiranda y Juan Esteban González Duque.

**Objetivo.** Construir un modelo de regresión que prediga `vote_average` (calificación promedio de los usuarios de TMDB, escala 0 a 10) usando solo características que se conocen antes o al momento del estreno de una película.

**Dataset.** The Movies Dataset (Rounak Banik, Kaggle), archivo `movies_metadata.csv`. El README del repositorio explica cómo descargarlo y dónde ubicarlo (`data/movies_metadata.csv`).

**Estructura del notebook.**

1. Carga de datos
2. Limpieza: filas corruptas y películas duplicadas
3. Análisis exploratorio (EDA)
4. Preparación de variables y fuga de información
5. División train/test y Pipeline de preprocesamiento
6. Modelos y evaluación
7. Guardado del modelo

El notebook está pensado para ejecutarse de principio a fin (Restart & Run All) con el kernel del entorno virtual del proyecto.

## 1. Carga de datos

In [1]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Semilla única para todo lo aleatorio del notebook (reproducibilidad)
SEMILLA = 42

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Estilo de gráficos: un solo color para una sola serie, rejilla discreta
COLOR_PRINCIPAL = "#2a78d6"
plt.rcParams.update({
    "figure.dpi": 90,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

El notebook vive en `fase-1/` y el CSV en `data/` (en la raíz del repositorio). La celda siguiente localiza la raíz tanto si el notebook se ejecuta desde `fase-1/` como desde la raíz.

In [2]:
RAIZ = Path.cwd().resolve()
if not (RAIZ / "data").is_dir():
    RAIZ = RAIZ.parent
RUTA_CSV = RAIZ / "data" / "movies_metadata.csv"

if not RUTA_CSV.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_CSV}. Descargar movies_metadata.csv de "
        "https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset "
        "y copiarlo en la carpeta data/ (ver README)."
    )

# low_memory=False: lee cada columna completa antes de inferir su tipo
df_raw = pd.read_csv(RUTA_CSV, low_memory=False)
print(f"Archivo: {RUTA_CSV.name}")
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")

Archivo: movies_metadata.csv
Filas: 45,466 | Columnas: 24


In [3]:
df_raw.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


Tipos con los que pandas leyó cada columna. Varias columnas que deberían ser numéricas (`budget`, `id`, `popularity`) llegan como texto: es la primera señal de que hay filas mal formadas en el CSV.

In [4]:
df_raw.dtypes.to_frame("tipo")

,tipo
adult,str
belongs_to_collection,str
budget,str
genres,str
homepage,str
id,str
imdb_id,str
original_language,str
original_title,str
overview,str


## 2. Limpieza: filas corruptas y películas duplicadas

### 2.1 Filas corruptas por parsing

Algunas filas del CSV tienen comas o saltos de línea mal escapados dentro del texto de `overview`. Al leerlas, los valores quedan desplazados una o varias columnas: en `adult` aparece un pedazo de sinopsis, en `budget` una ruta de póster y en `id` una fecha. Una forma directa de detectarlas es que `id` debe ser un entero; en esas filas no lo es.

In [5]:
id_numerico = pd.to_numeric(df_raw["id"], errors="coerce")
filas_corruptas = df_raw[id_numerico.isna()]

print(f"Filas con id no numérico: {len(filas_corruptas)}")
filas_corruptas[["adult", "belongs_to_collection", "budget", "id", "vote_average"]]

Filas con id no numérico: 3


,adult,belongs_to_collection,budget,id,vote_average
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,1997-08-20,NaN
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,2012-09-29,NaN
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,2014-01-01,NaN


Las filas detectadas son exactamente las que tienen valores desplazados: `adult` contiene texto de una sinopsis, `belongs_to_collection` un número (que en realidad es `popularity`) y `budget` la ruta de un póster. No se pueden reconstruir con seguridad, así que se eliminan.

In [6]:
df = df_raw[id_numerico.notna()].copy()
assert len(df) == len(df_raw) - len(filas_corruptas)
# Tras quitar las filas corruptas, adult solo tiene los valores esperados
assert set(df["adult"].unique()) == {"True", "False"}
print(f"Filas eliminadas: {len(filas_corruptas)} | Filas restantes: {len(df):,}")

Filas eliminadas: 3 | Filas restantes: 45,463


Cada una de esas líneas partidas deja además una fila incompleta justo antes: la primera mitad de la película, cuyos datos continúan en la fila corrupta. Esas filas tienen casi todas las columnas vacías, incluida la variable objetivo `vote_average`, así que tampoco se pueden usar.

In [7]:
mitades = df[df["vote_average"].isna()]
# Son exactamente las filas anteriores a cada fila corrupta
assert (mitades.index == filas_corruptas.index - 1).all()
print(f"Filas incompletas (primera mitad de la línea partida): {len(mitades)}")
print("Columnas con dato en cada una:", mitades.notna().sum(axis=1).tolist(), "de", df.shape[1])
mitades[["id", "title", "release_date", "runtime", "vote_average", "vote_count"]]

Filas incompletas (primera mitad de la línea partida): 3
Columnas con dato en cada una: [8, 10, 8] de 24


,id,title,release_date,runtime,vote_average,vote_count
19729,82663,NaN,NaN,NaN,NaN,NaN
29502,122662,NaN,NaN,NaN,NaN,NaN
35586,249260,NaN,NaN,NaN,NaN,NaN


In [8]:
df = df.drop(index=mitades.index)
assert df["vote_average"].notna().all()
print(f"Filas restantes: {len(df):,}")

Filas restantes: 45,460


Con las filas corruptas fuera, las columnas numéricas que pandas había leído como texto ya se pueden convertir a su tipo real.

In [9]:
df["id"] = df["id"].astype("int64")
df["budget"] = pd.to_numeric(df["budget"], errors="raise")
df["popularity"] = pd.to_numeric(df["popularity"], errors="raise")
df[["id", "budget", "popularity", "revenue", "runtime", "vote_average", "vote_count"]].dtypes.to_frame("tipo")

,tipo
id,int64
budget,int64
popularity,float64
revenue,float64
runtime,float64
vote_average,float64
vote_count,float64


### 2.2 Películas duplicadas

`id` es el identificador de la película en TMDB, así que debería ser único. Se revisa si hay películas repetidas y, en ese caso, en qué columnas difieren las copias.

In [10]:
repetidas = df[df["id"].duplicated(keep=False)]
print(f"Filas con id repetido: {len(repetidas)} ({repetidas['id'].nunique()} películas distintas)")

columnas_que_difieren = sorted({
    col
    for _, grupo in repetidas.groupby("id")
    for col in df.columns
    if grupo[col].astype(str).nunique() > 1
})
print("Columnas en las que difieren las copias:", columnas_que_difieren)

Filas con id repetido: 59 (29 películas distintas)
Columnas en las que difieren las copias: ['popularity', 'vote_count']


Las copias son la misma película: solo cambian `popularity` y `vote_count`, que además quedan excluidas como predictoras (sección 4). Se conserva la primera aparición de cada `id`. Además de limpiar, esto evita un problema de fuga de información: si se dejaran duplicados, una misma película podría caer en train y en test, y el modelo se evaluaría sobre ejemplos que ya vio al entrenar.

In [11]:
filas_antes = len(df)
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
assert df["id"].is_unique
print(f"Duplicados eliminados: {filas_antes - len(df)} | Filas restantes: {len(df):,}")

Duplicados eliminados: 30 | Filas restantes: 45,430
